In [ ]:
!mkdir -p megha data checkpoints
!pip install transformers tokenizers accelerate huggingface_hub


In [ ]:
%%writefile megha/__init__.py



In [ ]:
%%writefile megha/config.py
from dataclasses import dataclass

@dataclass
class MeghaConfig:
    vocab_size: int = 4000
    max_seq_len: int = 256
    d_model: int = 256
    n_layers: int = 6
    n_heads: int = 4
    dropout: float = 0.1
    batch_size: int = 4       # Small batch: works even with small per-level datasets
    learning_rate: float = 5e-4
    epochs: int = 10          # 10 epochs for proper loss convergence



In [ ]:
%%writefile megha/model.py
import torch
import torch.nn as nn
from .config import MeghaConfig

class MeghaBlock(nn.Module):
    def __init__(self, config: MeghaConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=config.d_model, 
            num_heads=config.n_heads, 
            dropout=config.dropout,
            batch_first=True
        )
        self.ln_2 = nn.LayerNorm(config.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model),
            nn.Dropout(config.dropout)
        )

    def forward(self, x, attention_mask=None):
        B, T, C = x.shape
        attn_mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        
        attn_out, _ = self.attn(
            self.ln_1(x), self.ln_1(x), self.ln_1(x), 
            attn_mask=attn_mask, need_weights=False, is_causal=True
        )
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x

class MeghaModel(nn.Module):
    def __init__(self, config: MeghaConfig):
        super().__init__()
        self.config = config
        
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        
        self.blocks = nn.Sequential(*[MeghaBlock(config) for _ in range(config.n_layers)])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        
        # Weight tying
        self.token_emb.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.config.vocab_size), targets.view(-1))
            
        return logits, loss
        
    def generate(self, idx, max_new_tokens, temperature=0.7, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            
            if top_k is not None and top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
                
            probs = torch.nn.functional.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



In [ ]:
%%writefile megha/tokenizer.py
import os
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from .config import MeghaConfig

class MeghaTokenizer:
    def __init__(self, config: MeghaConfig):
        self.config = config
        self.tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
        self.tokenizer.pre_tokenizer = Whitespace()
        self.trainer = BpeTrainer(
            vocab_size=config.vocab_size, 
            special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
        )
        
    def train_from_iterator(self, iterator):
        self.tokenizer.train_from_iterator(iterator, self.trainer)
        
    def save(self, path):
        self.tokenizer.save(path)
        
    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)
        
    def encode(self, text):
        return self.tokenizer.encode(text).ids
        
    def decode(self, ids):
        return self.tokenizer.decode(ids)

if __name__ == "__main__":
    # Script to train the tokenizer on generated curriculum data
    print("Training Tokenizer...")
    config = MeghaConfig()
    megha_tok = MeghaTokenizer(config)
    
    # 1. Extract text into an iterator from all available curriculum files
    def text_iterator():
        import glob
        files = glob.glob("data/level_*_curriculum.json")
        for file_path in files:
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    dataset = json.load(f)
                    for item in dataset:
                        if isinstance(item, dict) and "text" in item and item["text"]:
                            yield item["text"]
            except Exception:
                pass
            
    # 2. Train
    megha_tok.train_from_iterator(text_iterator())
    
    # 4. Save
    os.makedirs("data", exist_ok=True)
    megha_tok.save("data/tokenizer.json")
    print(f"Tokenizer trained and saved to data/tokenizer.json with vocab size: {megha_tok.tokenizer.get_vocab_size()}")
    
    # Quick Test
    test_text = "The computer is on."
    encoded = megha_tok.encode(test_text)
    print(f"\nTest string: '{test_text}'")
    print(f"Encoded IDs: {encoded}")
    print(f"Decoded: '{megha_tok.decode(encoded)}'")



In [ ]:
%%writefile megha/dataset.py
import json
import glob
import torch
import random
from torch.utils.data import Dataset, DataLoader
from .tokenizer import MeghaTokenizer
from .config import MeghaConfig
import os

class MeghaDataset(Dataset):
    def __init__(self, all_texts: list, tokenizer: MeghaTokenizer, config: MeghaConfig):
        self.config = config
        self.tokenizer = tokenizer
        
        # EOS separator between each Q&A example
        eos_tokens = self.tokenizer.encode("<|endoftext|>")
        if not eos_tokens:
            eos_tokens = [0]
            
        all_tokens = []
        for text in all_texts:
            if not text:
                continue
            tokens = self.tokenizer.encode(text)
            if tokens:
                all_tokens.extend(tokens)
                all_tokens.extend(eos_tokens)
            
        # Pad if still too small
        while len(all_tokens) <= self.config.max_seq_len + 1:
            all_tokens.extend(eos_tokens * 10)
            
        self.data = torch.tensor(all_tokens, dtype=torch.long)
        # 50% stride: good balance between coverage and overfitting prevention
        self.stride = max(1, self.config.max_seq_len // 2)
        
    def __len__(self):
        return max(1, (len(self.data) - self.config.max_seq_len - 1) // self.stride)
        
    def __getitem__(self, idx):
        start_idx = idx * self.stride
        x = self.data[start_idx : start_idx + self.config.max_seq_len]
        y = self.data[start_idx + 1 : start_idx + self.config.max_seq_len + 1]
        return x, y


def load_texts_from_file(data_path: str) -> list:
    """Load all Q&A texts from a single curriculum JSON file."""
    if not os.path.exists(data_path):
        return []
    try:
        with open(data_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        texts = []
        for item in raw_data:
            text = item.get("text", "")
            if text and len(text.strip()) > 5:
                texts.append(text.strip())
        return texts
    except Exception as e:
        print(f"Warning: Could not load {data_path}: {e}")
        return []


def get_combined_dataloader(tokenizer_path: str, config: MeghaConfig, data_dir: str = "data"):
    """
    MIXED TRAINING: Load ALL levels' data, shuffle together, train ONE model.
    This prevents Catastrophic Forgetting.
    """
    tokenizer = MeghaTokenizer(config)
    if os.path.exists(tokenizer_path):
        tokenizer.load(tokenizer_path)
    else:
        raise FileNotFoundError(f"Tokenizer not found at {tokenizer_path}. Run tokenizer.py first.")
    
    # Load all curriculum files
    all_texts = []
    files = sorted(glob.glob(f"{data_dir}/level_*_curriculum.json"))
    
    for fpath in files:
        texts = load_texts_from_file(fpath)
        all_texts.extend(texts)
        print(f"  Loaded {len(texts)} examples from {os.path.basename(fpath)}")
    
    if not all_texts:
        raise ValueError("No training data found! Run data_gen.py first.")
    
    # SHUFFLE: mix all levels together so model learns all topics uniformly
    random.shuffle(all_texts)
    print(f"\nTotal training examples (all levels combined): {len(all_texts)}")
    
    dataset = MeghaDataset(all_texts, tokenizer, config)
    print(f"Total dataset chunks: {len(dataset)}")
    
    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        drop_last=False
    )
    return dataloader, tokenizer


def get_dataloader(data_path: str, tokenizer_path: str, config: MeghaConfig):
    """Single-level loader (kept for backward compatibility)."""
    tokenizer = MeghaTokenizer(config)
    if os.path.exists(tokenizer_path):
        tokenizer.load(tokenizer_path)
    else:
        raise FileNotFoundError(f"Tokenizer not found at {tokenizer_path}. Run tokenizer.py first.")
    
    texts = load_texts_from_file(data_path)
    dataset = MeghaDataset(texts, tokenizer, config)
    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        drop_last=False
    )
    return dataloader, tokenizer



In [ ]:
%%writefile megha/data_gen.py
import json
import argparse
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Prompts for different levels of curriculum
LEVEL_PROMPTS = {
    0: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 0 - Basic English Grammar and Vocabulary.
Generate 50 simple Q&A examples covering basic sentence structure, nouns, verbs, and pronouns.
Format each example STRICTLY as: "Q: <simple question>\nA: <simple answer>".
Format the output STRICTLY as a JSON array of objects, each with a "text" field.
Example: [{"text": "Q: Is the cat sleeping?\nA: Yes, the cat is sleeping on the bed."}]
Output nothing but the JSON array. Do not include markdown blocks.""",
    
    1: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 1 - General Knowledge & Basic Reasoning.
Generate 50 Q&A examples covering numbers, comparison, time, input/output, and basic cause-effect reasoning.
Format each example STRICTLY as: "Q: <question>\nA: <clear answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    2: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 2 - Computer Fundamentals.
Generate 50 Q&A examples covering CPU, RAM, Storage (HDD vs SSD), Operating Systems (kernel, processes), and basic computing.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    3: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 3 - Linux Operating System.
Generate 50 Q&A examples covering Linux commands (chmod, chown, ls, grep, ps, systemctl), filesystem (/etc, /var), and file permissions.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    4: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 4 - Networking.
Generate Q&A examples covering TCP/IP, OSI model layers, DNS, CIDR subnetting, HTTP status codes (200, 404, 502), and common ports (22, 80, 443).
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    5: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 5 - Cloud Computing Fundamentals.
Generate Q&A examples covering virtualization, Cloud models (IaaS, PaaS, SaaS), deployment models (public, private, hybrid), and high availability.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    6: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 6 - AWS Core.
Generate Q&A examples covering Amazon EC2, S3 bucket storage, IAM roles and policies, VPC, and RDS databases.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    7: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 7 - Docker & Containers.
Generate Q&A examples covering Docker containers, images, Dockerfile instructions (FROM, RUN, CMD, COPY), docker build, docker run, and volumes.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    8: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 8 - Kubernetes.
Generate Q&A examples covering K8s pods, deployments, services (ClusterIP, NodePort), Ingress, and replica sets.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    9: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 9 - DevOps & CI/CD.
Generate Q&A examples covering Git commands, CI/CD pipelines, and Infrastructure as Code (Terraform).
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    10: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 10 - Cloud Security.
Generate Q&A examples covering authentication, IAM policies, why public S3 buckets are dangerous, Zero Trust, and KMS encryption keys.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    11: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 11 - Cloud Troubleshooting.
Generate troubleshooting Q&A examples: symptoms, diagnosis, and fix (e.g. 502 Bad Gateway cause and fix, EC2 unreachable cause and fix, S3 AccessDenied).
Format each example STRICTLY as: "Q: <troubleshooting question>\nA: <clear diagnostic and resolution steps>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    12: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 12 - Cloud Architecture.
Generate Q&A examples covering Highly Available designs, Load Balancer + Auto Scaling, and Serverless API architectures.
Format each example STRICTLY as: "Q: <architectural question>\nA: <clear architectural design explanation>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    13: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 13 - Cloud Reasoning.
Generate scenario-based Q&A examples analyzing traffic spikes, failover strategies, and database bottlenecks.
Format each example STRICTLY as: "Q: <scenario question>\nA: <logical step-by-step reasoning and solution>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    14: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 14 - CloudOps Multi-step Problem Solving.
Generate advanced Q&A examples showing step-by-step CloudOps problem resolution for Linux, AWS, Docker, and Kubernetes incidents.
Format each example STRICTLY as: "Q: <incident question>\nA: Identify symptoms -> Collect evidence -> Form hypothesis -> Test and Fix -> Verify.".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks."""
}

def generate_curriculum_real(level: int):
    print(f"Loading Qwen model for Level {level} curriculum generation...")
    # Upgraded Teacher to 3 Billion Parameters for much smarter data generation
    model_id = "Qwen/Qwen2.5-3B-Instruct"  
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    prompt = LEVEL_PROMPTS.get(level, LEVEL_PROMPTS[0])
    
    messages = [
        {"role": "system", "content": "You are a highly structured data generation AI."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    all_data = []
    target_examples = 300   # 300 Q&A pairs per level = 4500 total across 15 levels
    batch_size = 50         # Qwen generates 50 at a time (6 batches per level)
    
    print(f"Teacher is generating {target_examples} examples for Level {level} (in batches)...")
    
    iterations = target_examples // batch_size
    for i in range(iterations):
        print(f"Generating batch {i+1}/{iterations}...")
        try:
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=2048,
                temperature=0.8,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
            
            generated_ids = [
                output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]
            
            response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
            # Parse JSON
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0]
            elif "```" in response:
                response = response.split("```")[1].split("```")[0]
                
            clean_response = response.strip()
            data = None
            try:
                data = json.loads(clean_response, strict=False)
            except Exception:
                # Fallback: Robust regex extraction if LLM introduces unescaped quotes/newlines
                import re
                matches = re.findall(r'"text"\s*:\s*"(.*?)"', clean_response, re.DOTALL)
                if matches:
                    data = [{"text": m.replace('\\n', '\n').strip()} for m in matches]
                else:
                    raise
            
            # Ensure it's a list
            if isinstance(data, list):
                all_data.extend([d for d in data if isinstance(d, dict) and "text" in d and d["text"]])
            elif isinstance(data, dict) and "text" in data:
                all_data.append(data)
                
            print(f"Current total for Level {level}: {len(all_data)} examples.")
            
        except Exception as e:
            print(f"Batch {i+1} failed to parse or generate, skipping. Error: {e}")
            
    print(f"Successfully generated {len(all_data)} high-quality examples for Level {level}!")
    return all_data

def generate_curriculum_dummy(level: int):
    print(f"Generating DUMMY curriculum for Level {level} (Local PC Test)...")
    simulated_response = [
        {"text": "The computer is on."},
        {"text": "A network connects devices."},
        {"text": "She types on the keyboard."},
        {"text": "Data is stored in memory."},
        {"text": "He clicks the mouse."}
    ] * 100
    return simulated_response

def save_curriculum(data, level):
    os.makedirs("data", exist_ok=True)
    file_path = f"data/level_{level}_curriculum.json"
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)
    print(f"Curriculum saved to {file_path} successfully!")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Generate MEGHA Curriculum using Qwen")
    parser.add_argument("--level", type=int, default=0, help="Curriculum level to generate")
    parser.add_argument("--real", action="store_true", help="Use actual HuggingFace Qwen model (requires GPU)")
    args = parser.parse_args()
    
    if args.real:
        generated_data = generate_curriculum_real(args.level)
    else:
        generated_data = generate_curriculum_dummy(args.level)
        
    save_curriculum(generated_data, args.level)



In [ ]:
%%writefile megha/train.py
import torch
import torch.optim as optim
from .model import MeghaModel
from .config import MeghaConfig
from .dataset import get_combined_dataloader
import time
import os

def train_all():
    """
    MIXED TRAINING: Train ONE model on ALL 15 levels' data shuffled together.
    This eliminates Catastrophic Forgetting completely.
    """
    print("=" * 50)
    print("MEGHA MIXED TRAINING — All Levels Combined")
    print("=" * 50)
    
    config = MeghaConfig()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model = MeghaModel(config).to(device)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model Parameters: {num_params / 1e6:.2f} M\n")
    
    tokenizer_path = "data/tokenizer.json"
    
    try:
        dataloader, tokenizer = get_combined_dataloader(tokenizer_path, config)
    except (FileNotFoundError, ValueError) as e:
        print(f"ERROR: {e}")
        return
    
    total_batches = len(dataloader)
    if total_batches == 0:
        print("ERROR: Dataloader has 0 batches. Check your data files.")
        return
    
    print(f"\nBatches per epoch: {total_batches}")
    print(f"Epochs: {config.epochs}")
    print(f"Total training steps: {total_batches * config.epochs}\n")
    
    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)
    
    # Cosine LR scheduler for smooth convergence
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_batches * config.epochs
    )
    
    model.train()
    global_step = 0
    for epoch in range(config.epochs):
        print(f"\n--- Epoch {epoch+1}/{config.epochs} ---")
        epoch_loss = 0.0
        
        for step, (x, y) in enumerate(dataloader):
            t0 = time.time()
            
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            logits, loss = model(x, targets=y)
            loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            scheduler.step()
            
            dt = time.time() - t0
            epoch_loss += loss.item()
            global_step += 1
            
            if step % 20 == 0 or step == total_batches - 1:
                print(f"Step {global_step} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.2e} | Time: {dt*1000:.1f}ms")
        
        avg_loss = epoch_loss / total_batches
        print(f"Epoch {epoch+1} avg loss: {avg_loss:.4f}")
    
    # Save the final unified checkpoint
    os.makedirs("checkpoints", exist_ok=True)
    final_path = "checkpoints/megha_final.pt"
    torch.save(model.state_dict(), final_path)
    
    # Also save as level_14 for backwards compatibility with evaluate.py fallback
    torch.save(model.state_dict(), "checkpoints/megha_level_14.pt")
    
    print(f"\n{'='*50}")
    print(f"Training complete! Final model saved to {final_path}")
    print(f"Total steps trained: {global_step}")
    print(f"{'='*50}")


# Keep old function for backward compatibility
def train_level(level: int):
    """Deprecated: use train_all() instead."""
    print(f"Note: train_level() is deprecated. Use train_all() for better results.")
    train_all()


if __name__ == "__main__":
    train_all()



In [ ]:
%%writefile megha/evaluate.py
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from .model import MeghaModel
from .config import MeghaConfig
from .tokenizer import MeghaTokenizer
import os

def run_evaluation():
    print("Starting MEGHA Evaluation Phase (Level 15)...")
    
    # Load MEGHA
    config = MeghaConfig()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    megha_model = MeghaModel(config).to(device)
    
    # Load the best available checkpoint (Level 14 first, then fallback)
    loaded = False
    for lvl in range(14, -1, -1):
        ckpt_path = f"checkpoints/megha_level_{lvl}.pt"
        if os.path.exists(ckpt_path):
            megha_model.load_state_dict(torch.load(ckpt_path, map_location=device))
            print(f"Loaded MEGHA from {ckpt_path} (Level {lvl})")
            loaded = True
            break
    if not loaded:
        print("CRITICAL: No checkpoint found at all! Evaluation will use random weights.")
        
    megha_model.eval()
    megha_tok = MeghaTokenizer(config)
    megha_tok.load("data/tokenizer.json")
    
    # Load Qwen (Teacher)
    print("Loading Teacher (Qwen 3B) for grading...")
    teacher_id = "Qwen/Qwen2.5-3B-Instruct"
    teacher_tok = AutoTokenizer.from_pretrained(teacher_id)
    teacher = AutoModelForCausalLM.from_pretrained(
        teacher_id, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    
    # 5 Sample Questions covering the syllabus
    test_questions = {
        "Level 3 (Linux)": "What is the command to change file permissions in Linux?",
        "Level 6 (AWS)": "What is Amazon EC2 used for?",
        "Level 7 (Docker)": "What does a Dockerfile do?",
        "Level 10 (Security)": "Why should you not store AWS access keys in a public S3 bucket?",
        "Level 11 (Troubleshooting)": "If a website returns a 502 error, what could be the problem?"
    }
    
    results = {}
    
    for topic, question in test_questions.items():
        print(f"\n[Testing {topic}] Question: {question}")
        
        # 1. MEGHA generates an answer
        prompt = f"Q: {question}\nA:"
        input_ids = megha_tok.encode(prompt)
        
        # If tokenizer returns empty or very short, pad it safely
        if len(input_ids) == 0:
            input_ids = [0]
            
        x = torch.tensor([input_ids], dtype=torch.long).to(device)
        
        # Generate tokens using temperature=0.5 (focused, not random)
        with torch.no_grad():
            out_ids = megha_model.generate(x, max_new_tokens=50, temperature=0.5, top_k=40)
                    
        full_decoded = megha_tok.decode(out_ids[0].tolist())
        # Clean up the output string
        megha_answer = full_decoded.replace(prompt, "").replace("<|endoftext|>", "").strip()
        # Cut off at next question if it rambles
        if "Q:" in megha_answer:
            megha_answer = megha_answer.split("Q:")[0].strip()
            
        print(f"MEGHA's Answer: {megha_answer}")
        
        # 2. Qwen grades the answer
        grade_prompt = f"""You are grading an AI student's answer.
Question: {question}
Student's Answer: {megha_answer}
Rate the student's answer out of 100 based on accuracy and conceptual relevance. If partially correct or relevant keywords are used, award partial marks (e.g., 40 to 80). If completely wrong or total gibberish, award 0.
Output ONLY the numeric score (e.g., 75)."""
        
        messages = [
            {"role": "system", "content": "You are a strict grader. Output only a number between 0 and 100."},
            {"role": "user", "content": grade_prompt}
        ]
        
        text = teacher_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = teacher_tok([text], return_tensors="pt").to(teacher.device)
        
        generated_ids = teacher.generate(
            **model_inputs,
            max_new_tokens=10,
            temperature=0.1
        )
        
        generated_ids = [
            out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)
        ]
        
        score_text = teacher_tok.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        # Ensure score is just digits
        score_text = ''.join(filter(str.isdigit, score_text))
        if not score_text: score_text = "0"
            
        print(f"Teacher's Grade: {score_text}/100")
        results[topic] = score_text
        
    print("\n" + "="*40)
    print("MEGHA FINAL REPORT CARD")
    print("="*40)
    for topic, score in results.items():
        print(f"{topic}: {score}%")
    print("="*40)

if __name__ == "__main__":
    run_evaluation()



In [ ]:
!python megha/data_gen.py --level 0 --real
!python megha/data_gen.py --level 1 --real
!python megha/data_gen.py --level 2 --real
!python megha/data_gen.py --level 3 --real
!python megha/data_gen.py --level 4 --real
!python megha/data_gen.py --level 5 --real
!python megha/data_gen.py --level 6 --real
!python megha/data_gen.py --level 7 --real
!python megha/data_gen.py --level 8 --real
!python megha/data_gen.py --level 9 --real
!python megha/data_gen.py --level 10 --real
!python megha/data_gen.py --level 11 --real
!python megha/data_gen.py --level 12 --real
!python megha/data_gen.py --level 13 --real
!python megha/data_gen.py --level 14 --real
!python -m megha.tokenizer
!python -c "from megha.train import train_all; train_all()"
!python -m megha.evaluate
!cp -r checkpoints/* /kaggle/working/ || true
!cp -r data /kaggle/working/ || true
